In [0]:
from pyspark.sql import functions as F

SOURCE_PATH = dbutils.widgets.get("source_path")
TARGET_SCHEMA = dbutils.widgets.get("target_schema")

In [0]:

datasets = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}


In [0]:

available_files = {
    file.name
    for file in dbutils.fs.ls(SOURCE_PATH)
}

for table_name, file_name in datasets.items():
    if file_name in available_files:
        print(f"FOUND: {file_name}")
    else:
        print(f"MISSING: {file_name}")


FOUND: olist_customers_dataset.csv
FOUND: olist_orders_dataset.csv
FOUND: olist_order_items_dataset.csv
FOUND: olist_order_payments_dataset.csv
FOUND: olist_order_reviews_dataset.csv
FOUND: olist_products_dataset.csv
FOUND: olist_sellers_dataset.csv
FOUND: olist_geolocation_dataset.csv
FOUND: product_category_name_translation.csv


In [0]:

def load_dataset(file_name):

    file_path = SOURCE_PATH + file_name

    df = (
        spark.read
        .format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(file_path)
    )

    df = (
        df
        .withColumn("ingested_at", F.current_timestamp())
        .withColumn(
            "source_file",
            F.col("_metadata.file_path")
        )
    )

    return df


In [0]:

for table_name, file_name in datasets.items():

    target_table = f"{TARGET_SCHEMA}.{table_name}"

    if spark.catalog.tableExists(target_table):
        print(f"SKIPPED: {target_table} already exists")
        continue

    print(f"Processing: {file_name}")

    df = load_dataset(file_name)

    (
        df.write
        .format("delta")
        .mode("ignore")
        .saveAsTable(target_table)
    )

    print(f"SUCCESS: {target_table}")


Processing: olist_customers_dataset.csv
SUCCESS: ecommerce_dev.bronze.customers
SKIPPED: ecommerce_dev.bronze.orders already exists
Processing: olist_order_items_dataset.csv
SUCCESS: ecommerce_dev.bronze.order_items
Processing: olist_order_payments_dataset.csv
SUCCESS: ecommerce_dev.bronze.order_payments
SKIPPED: ecommerce_dev.bronze.order_reviews already exists
Processing: olist_products_dataset.csv
SUCCESS: ecommerce_dev.bronze.products
Processing: olist_sellers_dataset.csv
SUCCESS: ecommerce_dev.bronze.sellers
Processing: olist_geolocation_dataset.csv
SUCCESS: ecommerce_dev.bronze.geolocation
Processing: product_category_name_translation.csv
SUCCESS: ecommerce_dev.bronze.category_translation


In [0]:

for table_name in datasets:

    full_table_name = f"{TARGET_SCHEMA}.{table_name}"

    if spark.catalog.tableExists(full_table_name):

        df = spark.table(full_table_name)

        print(f"\nTABLE: {full_table_name}")
        print(f"RECORD COUNT: {df.count()}")

        df.printSchema()

        display(df.limit(5))



TABLE: ecommerce_dev.bronze.customers
RECORD COUNT: 99441
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,ingested_at,source_file
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,2026-11-12T16:58:26.309Z,s3://ecommerce-lakehouse-pipeline-dev/raw/olist_customers_dataset.csv
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,2026-11-12T16:58:26.309Z,s3://ecommerce-lakehouse-pipeline-dev/raw/olist_customers_dataset.csv
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,2026-11-12T16:58:26.309Z,s3://ecommerce-lakehouse-pipeline-dev/raw/olist_customers_dataset.csv
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,2026-11-12T16:58:26.309Z,s3://ecommerce-lakehouse-pipeline-dev/raw/olist_customers_dataset.csv
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,2026-11-12T16:58:26.309Z,s3://ecommerce-lakehouse-pipeline-dev/raw/olist_customers_dataset.csv



TABLE: ecommerce_dev.bronze.orders
RECORD COUNT: 99441
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- last_read_at: timestamp (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_name: string (nullable = true)



order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,last_read_at,source_file_path,source_file_name
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z,2026-10-24T08:37:21.026Z,s3://ecommerce-lakehouse-pipeline-dev/raw/olist_orders_dataset.csv,olist_orders_dataset.csv
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z,2026-10-24T08:37:21.026Z,s3://ecommerce-lakehouse-pipeline-dev/raw/olist_orders_dataset.csv,olist_orders_dataset.csv
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z,2026-10-24T08:37:21.026Z,s3://ecommerce-lakehouse-pipeline-dev/raw/olist_orders_dataset.csv,olist_orders_dataset.csv
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z,2026-10-24T08:37:21.026Z,s3://ecommerce-lakehouse-pipeline-dev/raw/olist_orders_dataset.csv,olist_orders_dataset.csv
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z,2026-10-24T08:37:21.026Z,s3://ecommerce-lakehouse-pipeline-dev/raw/olist_orders_dataset.csv,olist_orders_dataset.csv



TABLE: ecommerce_dev.bronze.order_items
RECORD COUNT: 112650
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,ingested_at,source_file
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35.000Z,58.9,13.29,2026-11-12T16:58:46.764Z,s3://ecommerce-lakehouse-pipeline-dev/raw/olist_order_items_dataset.csv
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13.000Z,239.9,19.93,2026-11-12T16:58:46.764Z,s3://ecommerce-lakehouse-pipeline-dev/raw/olist_order_items_dataset.csv
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18T14:48:30.000Z,199.0,17.87,2026-11-12T16:58:46.764Z,s3://ecommerce-lakehouse-pipeline-dev/raw/olist_order_items_dataset.csv
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15T10:10:18.000Z,12.99,12.79,2026-11-12T16:58:46.764Z,s3://ecommerce-lakehouse-pipeline-dev/raw/olist_order_items_dataset.csv
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13T13:57:51.000Z,199.9,18.14,2026-11-12T16:58:46.764Z,s3://ecommerce-lakehouse-pipeline-dev/raw/olist_order_items_dataset.csv


... (8 outputs left out)


TABLE: ecommerce_dev.bronze.geolocation
RECORD COUNT: 1000163
root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,ingested_at,source_file
1037,-23.54562128115268,-46.63929204800168,sao paulo,SP,2026-11-12T16:59:32.743Z,s3://ecommerce-lakehouse-pipeline-dev/raw/olist_geolocation_dataset.csv
1046,-23.546081127035535,-46.64482029837157,sao paulo,SP,2026-11-12T16:59:32.743Z,s3://ecommerce-lakehouse-pipeline-dev/raw/olist_geolocation_dataset.csv
1046,-23.54612896641469,-46.64295148361138,sao paulo,SP,2026-11-12T16:59:32.743Z,s3://ecommerce-lakehouse-pipeline-dev/raw/olist_geolocation_dataset.csv
1041,-23.5443921648681,-46.63949930627844,sao paulo,SP,2026-11-12T16:59:32.743Z,s3://ecommerce-lakehouse-pipeline-dev/raw/olist_geolocation_dataset.csv
1035,-23.541577961711493,-46.64160722329613,sao paulo,SP,2026-11-12T16:59:32.743Z,s3://ecommerce-lakehouse-pipeline-dev/raw/olist_geolocation_dataset.csv



TABLE: ecommerce_dev.bronze.category_translation
RECORD COUNT: 71
root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



product_category_name,product_category_name_english,ingested_at,source_file
beleza_saude,health_beauty,2026-11-12T16:59:45.299Z,s3://ecommerce-lakehouse-pipeline-dev/raw/product_category_name_translation.csv
informatica_acessorios,computers_accessories,2026-11-12T16:59:45.299Z,s3://ecommerce-lakehouse-pipeline-dev/raw/product_category_name_translation.csv
automotivo,auto,2026-11-12T16:59:45.299Z,s3://ecommerce-lakehouse-pipeline-dev/raw/product_category_name_translation.csv
cama_mesa_banho,bed_bath_table,2026-11-12T16:59:45.299Z,s3://ecommerce-lakehouse-pipeline-dev/raw/product_category_name_translation.csv
moveis_decoracao,furniture_decor,2026-11-12T16:59:45.299Z,s3://ecommerce-lakehouse-pipeline-dev/raw/product_category_name_translation.csv
